# 04 — Treinamento conjunto e comparação anual

Execute o notebook 03 antes. Aqui juntamos os registros de treino dos oito anos e comparamos A, A+B, A+C e A+B+C. O limiar é escolhido na validação conjunta e aplicado ao teste conjunto e ao teste de cada ano, para que as taxas anuais usem a mesma regra dentro de cada combinação.

A regressão logística é treinada em blocos com SGD, que atualiza o modelo à medida que recebe mais pedidos. A floresta em blocos combina árvores treinadas em partições menores com pedidos de vários anos. Todos os pedidos de treino entram no fluxo. A floresta em blocos é diferente da floresta do estudo de 2024, na qual cada árvore podia sortear a partir de toda a base de ajuste. Como o modelo conjunto aprende com pedidos de todos os anos, a comparação anual descreve mudanças dentro desse período. Um teste para um ano futuro exigiria reservar esse ano inteiro antes do ajuste.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    REPORTS = ROOT / 'reports/local/hmda_multiano'
else:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hmda-credit-fairness')
    REPORTS = ROOT / 'reports/colab/hmda_multiano'

sys.path.insert(0, str(ROOT / 'scripts'))
from hmda_multiano import YEARS, SEED, CONFIGS, MODELS, annual_datasets, disparity_trends, train_and_compare

PROCESSED = ROOT / 'data/processed/hmda_multiano'
REPORTS.mkdir(parents=True, exist_ok=True)
partitions = pd.read_csv(PROCESSED / 'particoes_por_ano.csv')
if tuple(sorted(partitions.ano.unique())) != YEARS:
    raise ValueError('Execute o notebook 03 para preparar todos os anos.')
datasets = annual_datasets(PROCESSED, YEARS)
display(partitions.pivot(index='ano', columns='particao', values='n'))

particao,ajuste,teste,validacao
ano,,,
2018,2558946,365555,731141
2019,2607091,372437,744888
2020,2853064,407579,815168
2021,3014377,430637,861242
2022,2498269,356893,713787
2023,2019483,288494,577005
2024,2037450,291060,582132
2025,2059753,294243,588512


In [ ]:
metrics, groups, scenarios = train_and_compare(datasets, partitions)
gaps = disparity_trends(groups)
keys = ['dimensao', 'algoritmo', 'configuracao']
changes = gaps.loc[gaps.ano.ne('conjunto')].copy()
changes['ano'] = changes['ano'].astype(int)
changes = changes.sort_values(keys + ['ano'])
for field in ['delta_recusa_observada_pp', 'delta_recusa_prevista_pp', 'delta_fpr_pp', 'delta_fnr_pp']:
    changes[f'variacao_anual_{field}'] = changes.groupby(keys)[field].diff()
for name, frame in [
    ('metricas_gerais', metrics.query("ano == 'conjunto'")),
    ('metricas_por_ano', metrics.query("ano != 'conjunto'")),
    ('metricas_por_grupo_e_ano', groups),
    ('disparidades_por_ano', gaps),
    ('mudancas_ano_a_ano', changes),
    ('perfis_fixos_por_ano', scenarios),
]:
    frame.to_csv(REPORTS / f'{name}.csv', index=False)

summary = {
    'anos': list(YEARS), 'campos_validos': 18, 'semente': SEED,
    'particoes': partitions.to_dict(orient='records'),
    'modelos': list(MODELS), 'configuracoes': list(CONFIGS),
}
(REPORTS / 'execucao.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
display(metrics.query("ano == 'conjunto'").round(4))

Ajustando configuração A


## Como ler a evolução das disparidades

Cada ponto mostra uma taxa ou diferença de um ano. A diferença em pontos percentuais é a taxa do grupo em destaque menos a taxa do grupo de referência. A recusa observada vem da decisão registrada. A recusa prevista vem do modelo. Uma queda na diferença mede redução da disparidade observada naquele recorte, sem medir diretamente discriminação ou estabelecer a causa da mudança. Pesos de composição dos grupos, instituições, características dos pedidos, dados ausentes e condições econômicas podem variar entre anos.

In [ ]:
annual = metrics.loc[metrics.ano.ne('conjunto')].copy()
annual['ano'] = annual['ano'].astype(int)
metric_labels = {
    'roc_auc': 'ROC-AUC',
    'pr_auc_recusa': 'PR-AUC para recusas',
    'acuracia_balanceada': 'Acurácia balanceada',
}
fig, axes = plt.subplots(len(MODELS), len(metric_labels), figsize=(14, 7), constrained_layout=True)
for row, model in zip(axes, MODELS, strict=True):
    model_data = annual.loc[annual.algoritmo.eq(model)]
    for (metric, label), axis in zip(metric_labels.items(), row, strict=True):
        model_data.pivot(index='ano', columns='configuracao', values=metric).plot(ax=axis, marker='o')
        axis.set_title(f'{model} · {label}', fontsize=9)
        axis.set(xticks=list(YEARS), xlabel='', ylabel='Pontuação')
        axis.tick_params(axis='x', labelrotation=45)
fig.savefig(REPORTS / 'desempenho_por_ano.png', dpi=170)
plt.show()

trends = gaps.loc[gaps.ano.ne('conjunto')].copy()
trends['ano'] = trends['ano'].astype(int)
for field, label, filename in [
    ('derived_race', 'Negros em relação a brancos', 'disparidade_racial'),
    ('derived_ethnicity', 'Latinos em relação a não latinos', 'disparidade_etnica'),
]:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    for model, axis in zip(MODELS, axes, strict=True):
        series = trends.loc[trends.dimensao.eq(field) & trends.algoritmo.eq(model)]
        predicted = series.pivot(index='ano', columns='configuracao', values='delta_recusa_prevista_pp')
        predicted.plot(ax=axis, marker='o')
        observed = series.loc[series.configuracao.eq('A')].set_index('ano')['delta_recusa_observada_pp']
        axis.plot(observed.index, observed.values, 'k--', label='Recusa observada')
        axis.axhline(0, color='gray', linewidth=0.8)
        axis.set(xticks=list(YEARS), ylabel='Diferença em pontos percentuais', xlabel='', title=model)
        axis.tick_params(axis='x', labelrotation=45)
        axis.legend()
    fig.suptitle(label)
    fig.savefig(REPORTS / f'{filename}_por_ano.png', dpi=170)
    plt.show()

race_income = groups.loc[
    groups.ano.ne('conjunto') & groups.dimensao.eq('raca_x_renda')
    & groups.configuracao.eq('A+B+C') & groups.grupo.str.startswith('Black or African American | ')
].copy()
race_income['faixa'] = race_income.grupo.str.split(' | ', regex=False).str[-1]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
series = [('Recusa observada', MODELS[0], 'recusa_observada'),
          *[(f'Recusa prevista · {model}', model, 'recusa_prevista') for model in MODELS]]
for axis, (label, model, metric) in zip(axes, series, strict=True):
    subset = race_income.loc[race_income.algoritmo.eq(model)]
    subset.pivot(index='ano', columns='faixa', values=metric).plot(ax=axis, marker='o')
    axis.yaxis.set_major_formatter(PercentFormatter(1))
    axis.set(title=label, xlabel='Ano', ylabel='Pedidos recusados (%)')
    axis.tick_params(axis='x', labelrotation=45)
fig.suptitle('Grupo Black or African American por renda do setor')
fig.savefig(REPORTS / 'recusas_negras_por_faixa_e_ano.png', dpi=170)
plt.show()